# Churn Model — Business Value Demo

This notebook shows how a business would actually *use* the churn model to
make money, not just the technical metrics.

We call the deployed Vertex AI model as a live service, run predictions
across our customer base, then simulate a retention campaign: find the
customers the model flags as churn risks, "intervene" with a retention
offer, and calculate the return on that campaign.

In [1]:
from google.cloud import aiplatform
import pandas as pd

PROJECT_ID = "churn-mlops-498321"
BUCKET_NAME = "churn-mlops-bucket-caston"
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

# Reference the existing deployed endpoint by its resource name
endpoint = aiplatform.Endpoint(
    "projects/434019816081/locations/us-central1/endpoints/5740286875583643648"
)
print("Connected to endpoint:", endpoint.display_name)

Connected to endpoint: churn-rf-model_endpoint


> **Note on the live endpoint:** This notebook calls a live Vertex AI
> endpoint serving the churn model. The endpoint was undeployed after
> this run to avoid ongoing compute costs, so the prediction cells below
> show preserved outputs rather than re-running live. The model itself
> remains in the Vertex AI Model Registry and the artifact is stored in
> GCS, so the endpoint can be redeployed at any time. See `churn_deployment`
> for the registration and deployment code.

## Part 1: Live Endpoint Demo

A real business application would call the deployed model as a service over
the network. Here we send our entire customer base (the full test set) to the
live Vertex AI endpoint and get churn predictions back — exactly how a
production app would consume the model.

In [3]:
# Load the full test set — this represents our current customer base
test_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/test.csv")

features = test_df.drop(columns=["Churn"])
actual = test_df["Churn"].tolist()

# The live endpoint has a request size limit, so send in batches
# and collect all predictions
def predict_in_batches(endpoint, instances, batch_size=100):
    preds = []
    for i in range(0, len(instances), batch_size):
        batch = instances[i:i + batch_size]
        result = endpoint.predict(instances=batch)
        preds.extend([int(p) for p in result.predictions])
    return preds

print(f"Sending {len(features)} customers to the live endpoint...")
instances = features.values.tolist()
predicted = predict_in_batches(endpoint, instances)

print(f"Done. Model flagged {sum(predicted)} customers as churn risks "
      f"out of {len(predicted)} total.")

Sending 1407 customers to the live endpoint...
Done. Model flagged 290 customers as churn risks out of 1407 total.


## Part 2: Retention Campaign Simulation

The model flagged 290 customers as churn risks. A retention team would now
intervene — offer each flagged customer a discount or credit to keep them.

We simulate this campaign with realistic business assumptions and calculate
the return on investment:

- **Customer lifetime value (LTV):** monthly charges × 12 months
- **Retention offer cost:** a fixed cost per flagged customer (the discount)
- **Offer success rate:** the % of true churners who accept the offer and stay

We compare the total cost of the campaign against the revenue it saves.

In [5]:
# Attach predictions and actual outcomes back to the customer data
customers = test_df.copy()
customers["predicted_churn"] = predicted
customers["actual_churn"] = actual

# Business assumptions
RETENTION_HORIZON = 12      # months we'd keep a saved customer
OFFER_COST = 50             # $ spent per flagged customer on retention offer
OFFER_SUCCESS_RATE = 0.30   # 30% of true churners accept the offer and stay

# The flagged customers — our campaign targets
flagged = customers[customers["predicted_churn"] == 1].copy()

# Cost: we make an offer to every flagged customer
total_campaign_cost = len(flagged) * OFFER_COST

# Revenue saved: only the flagged customers who WOULD have actually churned
# AND who accept the offer represent real saved revenue
true_churners_flagged = flagged[flagged["actual_churn"] == 1]
customers_saved = len(true_churners_flagged) * OFFER_SUCCESS_RATE
revenue_saved = (true_churners_flagged["MonthlyCharges"].sum()
                 * RETENTION_HORIZON * OFFER_SUCCESS_RATE)

net_value = revenue_saved - total_campaign_cost

print("=== Retention Campaign Results ===")
print(f"Customers flagged & targeted:    {len(flagged)}")
print(f"Of those, actual churners:       {len(true_churners_flagged)}")
print(f"Estimated customers saved (30%): {customers_saved:.0f}")
print(f"\nCampaign cost:   ${total_campaign_cost:,.2f}")
print(f"Revenue saved:   ${revenue_saved:,.2f}")
print(f"Net value:       ${net_value:,.2f}")
print(f"ROI:             {(net_value / total_campaign_cost * 100):.0f}%")

=== Retention Campaign Results ===
Customers flagged & targeted:    290
Of those, actual churners:       178
Estimated customers saved (30%): 53

Campaign cost:   $14,500.00
Revenue saved:   $49,123.44
Net value:       $34,623.44
ROI:             239%


## What These Results Mean

The model flagged **290 customers** as churn risks. The retention team sent
each a \$50 offer, for a total campaign cost of **\$14,500**.

Of those 290, **178 were genuinely going to churn** (≈61% precision). Applying
a realistic 30% offer-acceptance rate, the campaign retains an estimated
**53 customers**, preserving **\$49,123** in 12-month revenue.

**Net value: \$34,623 — a 239% return.**

The model isn't perfect — 112 flagged customers would have stayed anyway, so
those offers were "wasted." But the campaign is still highly profitable,
because a wasted offer (\$50) is trivial next to the value of retaining a real
customer (~\$930 over 12 months). This is the business case for tuning a churn
model toward **high recall**: casting a wide net and absorbing some wasted
offers is far cheaper than missing customers who were about to leave.

## Future Improvements

This project is a working v1. Several extensions would make it more
realistic and robust:

**Model the offer as an ongoing discount.** This demo treats the retention
offer as a one-time $50 credit. A real business might offer an ongoing
monthly discount instead — which would reduce revenue from every customer
who accepts, including the false positives who would have stayed at full
price anyway. Modeling that lost margin would give a more complete ROI
picture.

**Return churn probabilities, not just labels.** The model currently outputs
a binary will-churn / will-stay. Exposing the underlying probability would
let the business prioritize — focus expensive interventions on the customers
most likely to leave, and cheaper ones on borderline cases.

**Tune the decision threshold to the business.** Rather than a fixed 0.5
cutoff, the threshold could be set to optimize for the actual cost of a
wasted offer versus the value of a saved customer.

**Add model monitoring.** Track prediction drift and feature skew over time
so the team knows when the model needs retraining.

**Automate retraining.** A scheduled pipeline could retrain the model on
fresh data and redeploy automatically when performance stays above a
threshold.

**Use one-hot encoding for categorical features.** Label encoding implies a
false ordering for unordered categories (e.g., payment method). One-hot
encoding would represent these more correctly.

In [9]:
# Undeploy the model to stop endpoint billing
# The endpoint resource and registered model remain — only the running compute is removed
endpoint.undeploy_all()
print("Model undeployed. Endpoint billing stopped.")
print("The endpoint and registered model still exist and can be redeployed anytime.")

Undeploying Endpoint model: projects/434019816081/locations/us-central1/endpoints/5740286875583643648
Undeploy Endpoint model backing LRO: projects/434019816081/locations/us-central1/endpoints/5740286875583643648/operations/6481378251238277120
Endpoint model undeployed. Resource name: projects/434019816081/locations/us-central1/endpoints/5740286875583643648
Model undeployed. Endpoint billing stopped.
The endpoint and registered model still exist and can be redeployed anytime.
